In [ ]:
# Cell 1:  Setup and imports

import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import compute_all_iv, plot_default_rate_by_bin

In [ ]:
# Cell 2:  Load clean data, train/test split

df = pd.read_csv('../outputs/loan_book_clean.csv')

train = df[df["set"] == "train"].copy()
test = df[df["set"] == "test"].copy()

print(f"Train: {len(train)} rows, Test: {len(test)}")

In [ ]:
# Cell 3:  Compute IV for all features and create ranked table
exclude = ["applicant_id_hash", "default_flag", "set", "application_date"]

iv_summary = compute_all_iv(train, target="default_flag", exclude_cols=exclude)
print(iv_summary.to_string(index=False))

In [ ]:
# Cell 4:  Plot default rate by bin for top 8 features

# top_features = [
#     'interest_rate',
#     'months_since_last_delinquency',
#     'annual_income',
#     'age',
#     'num_delinquencies_2yr',
#     'months_since_oldest_account'
# ]

top_features = [
    'interest_rate', 'months_since_last_delinquency', 'annual_income',
    'age', 'num_delinquencies_2yr', 'months_since_oldest_account',
    'employment_length_years', 'dti_ratio'
]

for feat in top_features:
    plot_default_rate_by_bin(train, feat)

In [ ]:
# Cell 5:  Correlation check

corr = train[top_features].corr().round(2)
print(corr)

In [ ]:
# Cell 6:  Bivariate analysis

# Annual income and interest rate
train["income_group"] = pd.qcut(train["annual_income"], q=3, labels=["low", "medium", "high"])
train["rate_group"] = pd.qcut(train["interest_rate"], q=3, labels=["low", "medium", "high"])

income_rate_cross = train.groupby(["rate_group", "income_group"])["default_flag"].agg(
    default_rate="mean",
    count="count",
).reset_index()

print("Annual income and interest rate")
print(income_rate_cross.to_string(index=False))

# dti_ratio and annual income
train["dti_group"] = pd.qcut(train["dti_ratio"], q=3, labels=["low", "medium", "high"])

income_dti_cross = train.groupby(["dti_group", "income_group"])["default_flag"].agg(
    default_rate="mean",
    count="count",
).reset_index()

print("\nAnnual income and dti ratio")
print(income_dti_cross.to_string(index=False))

# Age and num_delinquencies_2yr
train["age_group"] = pd.qcut(train["age"], q=2, labels=["young", "old"])
train["deliq_2yr_group"] = pd.cut(train["num_delinquencies_2yr"],
                                  bins=[-1, 0, 2, 9], labels=["none", "some", "many"]
)

num_deliq_age_cross = train.groupby(["age_group", "deliq_2yr_group"])["default_flag"].agg(
    default_rate="mean",
    count="count",
).reset_index()

print("\nAge and num_delinquencies_2yr")
print(num_deliq_age_cross.to_string(index=False))




# EDA Findings Summary

## Top Features and Their Signal Strength

| Feature | IV Score | Strength |
|---------|----------|----------|
| interest_rate | 0.41 | Strong |
| months_since_last_delinquency | 0.40 | Strong |
| annual_income | 0.37 | Strong |
| age | 0.25 | Strong |
| num_delinquencies_2yr | 0.24 | Strong |
| employment_length_years | 0.16 | Medium |
| pct_accounts_current | 0.09 | Weak |
| dti_ratio | 0.09 | Weak |

### interest rate (IV 0.41):
This was our strongest predictor. The default rate increases steadily from 5% to 35%, and it is perfectly monotonic. Note, there is a circular nature between interest rates and defaulting. The bank assigns higher rates to applicants it already perceives as risky, so interest rate partly reflects the bank's own risk assessment rather than being purely independent information

### months_since_last_delinquency (IV 0.40):
This is the second strongest predictor. Delinquency is positively related to default risk, for example, recent delinquency usually means high default risk. Filled nulls with 999 instead of 0, because filling with 0 would imply the person was delinquent yesterday, which is the opposite of the truth. 999 acts as a sentinel value meaning never delinquent.

### annual_income (IV 0.37):
Another strong predictor for defaulting. The default rates are mostly monotonic, starting high at 35% for low earners and drops smoothly to around 6% for high earners. However, there is a small bump around the $39k-$52k range. WoE binning will be used ti handle this non-monotonicity

### age (IV 0.25):
Another strong predictor of defaulting. Age is negatively related to the default rate, this means that younger borrowers default significantly more, and the default rate is perfectly monotonic. Please Note, Age is a protected characteristic in many jurisdictions and its use in credit
models may require regulatory justification.

### num_delinquencies_2yr (IV 0.24):
Similar to months_since_last_delinquency, this is another strong predictor and it also positively related to default risk. The greater the number of recent delinquencies, the higher the default rate

### employment_length_years (IV 0.16), pct_accounts_current (IV 0.09), dti_ratio (IV 0.09):
**These are medium to weak predictors, but they are still worth including in modelling


## Features Dropped and Why
### months_since_oldest_account:
It has a high correlation with age (0.95). This means that these features contain nearly identical information, and keeping both would cause multicollinearity in logistic regression

### ever_delinquent:
Created during cleaning as I wanted a way to keep track of people who were delinquents. But later I realised that added nothing more on top of  months_since_last_delinquency, which already encodes whether someone was ever a delinquent and how long ago was it

### region, branch_code_id, application_dow, loan_purpose, home_ownership, email_domain_type, phone_verified:
They each had an IV of 0.02 or lower, therefore it is safe to conclude that they are no meaningful signals here.

## Bivariate Findings
### Income and Interest_rate:
The two features are negatively related to each other, there is a strong interaction between them. High interest rate amplifies the risk of low income om defaulting. This combination is worse than either feature alone would suggest, therefore a simple income and interest_rate ratio or product feature might capture this

### DTI and Income:
These two features are both negatively related to default rate, but there is weak interaction between the two and they move independently of each other. The spread of the default rate remains consistent across DTI levels. They don't amplify each other as much, and probably not worth creating a combined feature

### Age and Delinquencies
There is no strong interaction between the two features but both features influence the default rate significantly so they should be included in the model separately

## Feature Engineering Decisions
- Use WoE binning on annual_income to fix the non-monotonicity
- Create a income and interest_rate interaction feature
- All other strong features are already appear monotonic and may not require WoE transformation, though this will be confirmed during modelling
